# 30 - Build Silver Contracts

Materializes service-contract master, revenue, visit schedule, active-call, and equipment coverage data. Optional GoCanvas models are created only when their shortcuts exist.

In [1]:
# Attach the schema-enabled Lakehouse containing the Bronze shortcuts and
# make it the default Lakehouse before running this notebook.
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
METADATA_SCHEMA = "metadata"
COMPANY_KEY = "NAC"  # Change when processing another GP company database.

def q(identifier: str) -> str:
    return f"`{identifier.replace('`', '``')}`"

def sql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"

def table_exists(schema_name: str, table_name: str) -> bool:
    try:
        return spark.catalog.tableExists(f"{schema_name}.{table_name}")
    except Exception:
        return False

def require_tables(table_names: list[str], schema_name: str = BRONZE_SCHEMA) -> None:
    missing = [name for name in table_names if not table_exists(schema_name, name)]
    if missing:
        raise RuntimeError(
            f"Missing tables in {schema_name}: {', '.join(missing)}. "
            "Confirm the OneLake shortcuts and exact table names."
        )

company = sql_string(COMPANY_KEY)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(SILVER_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(GOLD_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(METADATA_SCHEMA)}")
print(f"Company: {COMPANY_KEY}; Bronze schema: {BRONZE_SCHEMA}")

StatementMeta(, 4556eac6-8799-4724-a305-1efa3d505552, 3, Finished, Available, Finished, False)

Company: NAC; Bronze schema: bronze


In [2]:
GOCANVAS_SCHEMA = "gocanvas"
GOCANVAS_LOOKBACK_DAYS = 30
require_tables(["SV00500", "SV00501", "SV00509", "SV00510", "SV00585"])

StatementMeta(, 4556eac6-8799-4724-a305-1efa3d505552, 4, Finished, Available, Finished, False)

## Contract master

The repository repeated `SV00501` for active and inactive contracts. This conversion uses `SV00500` for active and `SV00501` for inactive, matching its other contract-union logic.

In [4]:
from pyspark.sql import functions as F

contract_source_df = (
    spark.table(f"{BRONZE_SCHEMA}.SV00500")
        .withColumn("contract_status", F.lit("Active"))
        .unionByName(
            spark.table(f"{BRONZE_SCHEMA}.SV00501")
                .withColumn("contract_status", F.lit("Inactive")),
            allowMissingColumns=True
        )
)
contract_source_df.createOrReplaceTempView("_contract_source")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_contract
USING DELTA
AS
WITH billing_counts AS (
    SELECT TRIM(Contract_Number) AS contract_number,
           COUNT(*) AS number_of_billings
    FROM {q(BRONZE_SCHEMA)}.SV00510
    GROUP BY TRIM(Contract_Number)
)
SELECT
    {company} AS company_key,
    c.contract_status,
    TRIM(c.CUSTNMBR) AS customer_code,
    TRIM(c.ADRSCODE) AS location_code,
    TRIM(c.Contract_Number) AS contract_number,
    TRY_CAST(c.WSCONTSQ AS BIGINT) AS contract_sequence,
    TRIM(c.Contract_Description) AS contract_description,
    TRIM(c.SLPRSNID) AS sales_rep_code,
    CASE
        WHEN LOWER(TRIM(c.User_Define_1a)) LIKE 'n%' THEN FALSE
        WHEN LOWER(TRIM(c.User_Define_1a)) LIKE 'y%' THEN TRUE
    END AS materials_included_flag,
    c.Bill_Freq AS billing_frequency,
    CAST(c.Contract_Amount AS DECIMAL(18,2)) AS contract_amount,
    CASE WHEN c.WSContractRenewalValue <> c.Contract_Amount
         THEN CAST(c.WSContractRenewalValue AS DECIMAL(18,2)) END AS contract_renewal_amount,
    CAST(c.Amount_Billed AS DECIMAL(18,2)) AS amount_billed,
    CAST(c.Annual_Contract_Value AS DECIMAL(18,2)) AS annual_contract_value,
    TRY_CAST(c.Service_User_Define_10 AS DATE) AS salesforce_date,
    CAST(c.Contract_Start_Date AS DATE) AS start_date,
    CAST(c.Contract_Expiration_Date AS DATE) AS end_date,
    CASE WHEN c.Multiyear_Contract_Flag > 0 THEN TRUE ELSE FALSE END AS multiyear_flag,
    CAST(c.Forecast_Original_Labor AS DECIMAL(18,2)) AS forecast_original_labor,
    (COALESCE(c.Forecast_Orig_Labor1_Hrs, 0)
     + COALESCE(c.Forecast_Orig_Labor3_Hrs, 0)
     + COALESCE(c.Forecast_Orig_Labor4_Hrs, 0)
     + COALESCE(c.Forecast_Orig_Labor5_Hrs, 0)) / 100.0 AS forecast_original_hours,
    CAST(NULL AS DECIMAL(18,2)) AS forecast_original_material,
    CAST(c.Forecast_Original_Equip AS DECIMAL(18,2)) AS forecast_original_equipment,
    CAST(c.Forecast_Original_Subs AS DECIMAL(18,2)) AS forecast_original_subcontractor,
    CAST(c.Forecast_Original_Other AS DECIMAL(18,2)) AS forecast_original_other,
    CAST(NULL AS DECIMAL(18,2)) AS forecast_original_total,
    CAST(c.Forecast_Labor AS DECIMAL(18,2)) AS forecast_labor,
    (COALESCE(c.Forecast_Labor_1_Hours, 0)
     + COALESCE(c.Forecast_Labor_2_Hours, 0)
     + COALESCE(c.Forecast_Labor_3_Hours, 0)
     + COALESCE(c.Forecast_Labor_4_Hours, 0)
     + COALESCE(c.Forecast_Labor_5_Hours, 0)) / 100.0 AS forecast_hours,
    CAST(c.Forecast_Material AS DECIMAL(18,2)) AS forecast_material,
    CAST(c.Forecast_Equipment AS DECIMAL(18,2)) AS forecast_equipment,
    CAST(c.Forecast_Subs AS DECIMAL(18,2)) AS forecast_subcontractor,
    CAST(c.Forecast_Other AS DECIMAL(18,2)) AS forecast_other,
    CAST(c.Forecast_Total_Cost AS DECIMAL(18,2)) AS forecast_total_cost,
    CAST(c.Estimate_Labor AS DECIMAL(18,2)) AS estimate_labor,
    (COALESCE(c.Estimate_Labor_1_Hours, 0)
     + COALESCE(c.Estimate_Labor_2_Hours, 0)
     + COALESCE(c.Estimate_Labor_3_Hours, 0)
     + COALESCE(c.Estimate_Labor_4_Hours, 0)
     + COALESCE(c.Estimate_Labor_5_Hours, 0)) / 100.0 AS estimate_hours,
    CAST(c.Estimate_Material AS DECIMAL(18,2)) AS estimate_material,
    CAST(c.Estimate_Equipment AS DECIMAL(18,2)) AS estimate_equipment,
    CAST(c.Estimate_Subs AS DECIMAL(18,2)) AS estimate_subcontractor,
    CAST(c.Estimate_Other AS DECIMAL(18,2)) AS estimate_other,
    CAST(c.Estimate_Total_Cost AS DECIMAL(18,2)) AS estimate_total_cost,
    CAST(c.Actual_Labor AS DECIMAL(18,2)) AS actual_labor,
    CAST(c.Actual_Hours AS DOUBLE) / 100.0 AS actual_hours,
    CAST(c.Actual_Material AS DECIMAL(18,2)) AS actual_material,
    CAST(c.Actual_Equipment AS DECIMAL(18,2)) AS actual_equipment,
    CAST(c.Actual_Subs AS DECIMAL(18,2)) AS actual_subcontractor,
    CAST(c.Actual_Other AS DECIMAL(18,2)) AS actual_other,
    CAST(c.Actual_Total_Cost AS DECIMAL(18,2)) AS actual_total_cost,
    CAST(c.Actual_Contract_Earned AS DECIMAL(18,2)) AS actual_contract_earned,
    CAST(c.Actual_Gross_Profit AS DECIMAL(18,2)) AS actual_gross_profit,
    CAST(c.Actua_Revenue_Recognized AS DECIMAL(18,2)) AS actual_revenue_recognized,
    CAST(c.Actual_Billed AS DECIMAL(18,2)) AS actual_billed,
    c.Contract_Billing_Date AS contract_billing_date,
    c.Contract_Service_Date AS contract_service_date,
    CAST(c.YTD_Labor AS DECIMAL(18,2)) AS ytd_labor,
    CAST(c.YTD_Material AS DECIMAL(18,2)) AS ytd_material,
    CAST(c.YTD_Equipment AS DECIMAL(18,2)) AS ytd_equipment,
    CAST(c.YTD_Subcontractor AS DECIMAL(18,2)) AS ytd_subcontractor,
    CAST(c.YTD_Other AS DECIMAL(18,2)) AS ytd_other,
    CAST(c.YTD_Total_Cost AS DECIMAL(18,2)) AS ytd_total_cost,
    CAST(c.YTD_Billed AS DECIMAL(18,2)) AS ytd_billed,
    CAST(c.YTD_Contract_Earned AS DECIMAL(18,2)) AS ytd_contract_earned,
    CAST(c.YTD_Gross_Profit AS DECIMAL(18,2)) AS ytd_gross_profit,
    CAST(c.YTD_Revenue_Recognized AS DECIMAL(18,2)) AS ytd_revenue_recognized,
    CASE WHEN c.Cancel_Box > 0 THEN TRUE ELSE FALSE END AS cancelled_flag,
    NULLIF(TRIM(c.User_Define_3a), '') AS cancellation_code,
    CASE WHEN TRY_CAST(c.Service_User_Define_9 AS DATE) > DATE'1990-01-01'
         THEN TRY_CAST(c.Service_User_Define_9 AS DATE) END AS cancellation_date,
    CASE WHEN c.HOLD > 0 THEN TRUE ELSE FALSE END AS hold_flag,
    b.number_of_billings,
    CURRENT_TIMESTAMP() AS _processed_at
FROM _contract_source c
LEFT JOIN billing_counts b
  ON TRIM(c.Contract_Number) = b.contract_number
WHERE TRIM(c.Contract_Number) <> ''
""")

StatementMeta(, 4556eac6-8799-4724-a305-1efa3d505552, 6, Finished, Available, Finished, False)

DataFrame[]

## Revenue and scheduled maintenance

In [9]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.contract_revenue_schedule
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRIM(CUSTNMBR) AS customer_code,
    TRIM(ADRSCODE) AS location_code,
    TRIM(Contract_Number) AS contract_number,
    TRY_CAST(WSCONTSQ AS BIGINT) AS contract_sequence,
    CASE WHEN YEAR1 > 0 THEN YEAR1 END AS billed_year,
    CASE WHEN PERIODID > 0 THEN PERIODID END AS billed_period,
    CAST(DOCAMNT AS DECIMAL(18,2)) AS document_amount,
    CASE WHEN CAST(DATE1 AS DATE) > DATE'1900-01-01' THEN CAST(DATE1 AS DATE) END AS bill_date,
    CASE WHEN CAST(POSTDATE AS DATE) > DATE'1900-01-01' THEN CAST(POSTDATE AS DATE) END AS posted_date,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.SV00509
WHERE TRIM(Contract_Number) <> ''
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.contract_schedule
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRIM(CUSTNMBR) AS customer_code,
    TRIM(ADRSCODE) AS location_code,
    TRIM(Contract_Number) AS contract_number,
    TRY_CAST(WSCONTSQ AS BIGINT) AS contract_sequence,
    CAST(Contract_Year AS INT) AS contract_year,
    CAST(Schedule_Date AS DATE) AS scheduled_date,
    CAST(Estimate_Hours AS DOUBLE) / 100.0 AS estimated_hours,
    TRIM(Service_Call_ID) AS work_number,
    TRIM(Equipment_ID) AS equipment_code,
    TRIM(Task_code) AS task_code,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.SV00585
WHERE TRIM(Contract_Number) <> ''
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.contract_visit_pending
USING DELTA
AS
SELECT DISTINCT
    company_key,
    contract_number,
    scheduled_date,
    YEAR(scheduled_date) AS scheduled_year,
    MONTH(scheduled_date) AS scheduled_month,
    CONCAT(DATE_FORMAT(scheduled_date, 'yyyyMM'), contract_number) AS scheduled_contract_xref,
    estimated_hours,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.contract_schedule
WHERE task_code = 'HOURS' AND COALESCE(work_number, '') = ''
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.contract_service_call
USING DELTA
AS
SELECT
    company_key, customer_code, location_code, contract_number,
    work_number, scheduled_date, estimated_hours,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.contract_schedule
WHERE equipment_code = 'HOURS' AND COALESCE(work_number, '') <> ''
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.contract_schedule_equipment
USING DELTA
AS
SELECT DISTINCT
    company_key, contract_number, equipment_code, scheduled_date,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.contract_schedule
WHERE equipment_code NOT IN ('', 'HOURS')
""")

StatementMeta(, 4556eac6-8799-4724-a305-1efa3d505552, 11, Finished, Available, Finished, False)

DataFrame[]

## Optional GoCanvas completion data

Add shortcuts for the GoCanvas tables to the configured `gocanvas` schema. This section skips safely until every required list table is present. To be implemented by Oct 2026.

In [6]:
list_specs = [
    (1, "BoilerID_List_1", "Recommendations_List_1"),
    (2, "ChillerID_List_2", "Recommendations_List_2"),
    (3, "CellID_List_3", "Recommendations_List_3"),
    (4, "PumpID_List_4", "Recommendations_List_4"),
    (5, "AIRHandlingUnitID_List_5", "Recommendations_List_5"),
    (6, "FanCoilUnit_List_6", "Recommendations_List_6"),
    (7, "UnitID_List_7", "Recommendations_List_7"),
    (8, "SplitSystemID_List_8", "Recommendations_List_8"),
    (9, "DryCoolerID_List_9", "Recommendations_List_9"),
    (10, "ExhaustFanID_List_10", None),
    (11, "VDFID_List_11", "Recommendations_List_11"),
    (16, "AirCompressorID_List_16", "Recommendations_List_16"),
    (17, "AirDryerID_List_17", "Recommendations_List_17"),
    (18, "DuctUnitHeaterID_List_18", None),
    (19, "HeatExchangerID_List_19", "Recommendations_List_19"),
    (20, "HumidifierType_List_20", "Recommendations_List_20"),
]
required_gc = ["ScheduledPMSubmission"] + [f"scheduledpmsubmissionlist{n}" for n, _, _ in list_specs]
missing_gc = [name for name in required_gc if not table_exists(GOCANVAS_SCHEMA, name)]

if missing_gc:
    print("Skipping GoCanvas models. Missing shortcuts: " + ", ".join(missing_gc))
else:
    union_parts = []
    for number, equipment_column, recommendation_column in list_specs:
        recommendation = (
            f"TRIM({q(recommendation_column)})" if recommendation_column
            else "CAST(NULL AS STRING)"
        )
        union_parts.append(f"""
            SELECT CAST({q(f'SubmissionID_List_{number}')} AS STRING) AS submission_id,
                   TRIM({q(equipment_column)}) AS equipment_code,
                   {recommendation} AS equipment_recommendations
            FROM {q(GOCANVAS_SCHEMA)}.{q(f'scheduledpmsubmissionlist{number}')}
        """)
    equipment_union = " UNION ".join(union_parts)

    spark.sql(f"""
    CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_contract_equipment_worked
    USING DELTA
    AS
    WITH equipment AS ({equipment_union})
    SELECT
        {company} AS company_key,
        CAST(m.SubmissionID AS STRING) AS submission_id,
        TRIM(m.Username) AS username,
        CAST(m.SubmissionCreationDateTime AS DATE) AS submission_date,
        TRIM(m.WO_Number) AS work_number,
        TRIM(m.ProjectDescriptionD) AS project_description,
        e.equipment_code,
        e.equipment_recommendations,
        CURRENT_TIMESTAMP() AS _processed_at
    FROM {q(GOCANVAS_SCHEMA)}.ScheduledPMSubmission m
    LEFT JOIN equipment e
      ON CAST(m.SubmissionID AS STRING) = e.submission_id
    WHERE m.SubmissionCreationDateTime > CURRENT_TIMESTAMP() - INTERVAL {GOCANVAS_LOOKBACK_DAYS} DAYS
    """)

    spark.sql(f"""
    CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_contract_call_submission
    USING DELTA
    AS
    SELECT DISTINCT
        COALESCE(e.company_key, w.company_key) AS company_key,
        COALESCE(e.work_number, w.work_number) AS work_number,
        w.submission_id,
        CURRENT_TIMESTAMP() AS _processed_at
    FROM {q(SILVER_SCHEMA)}.service_call_equipment e
    RIGHT JOIN {q(SILVER_SCHEMA)}.service_contract_equipment_worked w
      ON e.company_key = w.company_key AND e.work_number = w.work_number
    """)
    print("Created optional GoCanvas Silver tables.")

StatementMeta(, 4556eac6-8799-4724-a305-1efa3d505552, 8, Finished, Available, Finished, False)

Skipping GoCanvas models. Missing shortcuts: ScheduledPMSubmission, scheduledpmsubmissionlist1, scheduledpmsubmissionlist2, scheduledpmsubmissionlist3, scheduledpmsubmissionlist4, scheduledpmsubmissionlist5, scheduledpmsubmissionlist6, scheduledpmsubmissionlist7, scheduledpmsubmissionlist8, scheduledpmsubmissionlist9, scheduledpmsubmissionlist10, scheduledpmsubmissionlist11, scheduledpmsubmissionlist16, scheduledpmsubmissionlist17, scheduledpmsubmissionlist18, scheduledpmsubmissionlist19, scheduledpmsubmissionlist20


## Review

In [7]:
for name in ["service_contract", "contract_revenue_schedule", "contract_schedule",
             "contract_visit_pending", "contract_service_call", "contract_schedule_equipment"]:
    print(f"{SILVER_SCHEMA}.{name}: {spark.table(f'{SILVER_SCHEMA}.{name}').count():,} rows")

StatementMeta(, 4556eac6-8799-4724-a305-1efa3d505552, 9, Finished, Available, Finished, False)

silver.service_contract: 1,993 rows
silver.contract_revenue_schedule: 12,157 rows
silver.contract_schedule: 2,195,355 rows
silver.contract_visit_pending: 7,794 rows
silver.contract_service_call: 3,656 rows
silver.contract_schedule_equipment: 178,193 rows
